# Website Silver Layer: Fact Tables & dbt Model Generation

Transformed bronze website data into silver-layer fact tables, validated data quality and referential integrity, and generated dbt staging and marts model files for downstream analytics.

## 1. Building Silver Fact Tables from Bronze Website Data

Transformed three bronze tables into deduplicated silver fact tables:

* `fact_web_session` — sessions with source, medium, device, and campaign attribution
* `fact_web_page_view` — page views linked to sessions via `session_id`
* `fact_web_event` — interaction events (search, property view, booking, purchase) linked to sessions

Used a `latest_record()` helper to deduplicate by business key, keeping the most recent record per `updated_at` and `_airbyte_extracted_at`. Cast timestamps to proper types before writing to the `voyage.silver` schema.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ---------------------------------------------------------
# Helper for keeping the latest version of each record
# ---------------------------------------------------------

def latest_record(df, business_key):
    window = (
        Window
        .partitionBy(business_key)
        .orderBy(
            F.col("updated_at").desc(),
            F.col("_airbyte_extracted_at").desc(),
        )
    )

    return (
        df
        .withColumn("_row_number", F.row_number().over(window))
        .filter(F.col("_row_number") == 1)
        .drop("_row_number")
    )


# ---------------------------------------------------------
# 1. Web Sessions  (bronze --> silver fact table)
# ---------------------------------------------------------

bronze_web_sessions = spark.table("voyage.bronze.website_sessions")

web_sessions = (
    bronze_web_sessions
    .withColumn("session_start", F.to_timestamp("session_start"))
    .withColumn("session_end",   F.to_timestamp("session_end"))
    .withColumn("updated_at",     F.to_timestamp("updated_at"))
)

silver_web_sessions = latest_record(web_sessions, "session_id").select(
    "campaign_id",
    "session_id",
    "tenant_id",
    "user_id",
    "device",
    "country",
    "landing_page",
    "medium",
    "source",
    "session_start",
    "session_end",
    "updated_at",
)

(
    silver_web_sessions.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.fact_web_session")
)


# ---------------------------------------------------------
# 2. Web Page Views  (bronze → silver fact table)
# ---------------------------------------------------------

bronze_web_page_views = spark.table("voyage.bronze.website_page_views")

web_page_views = (
    bronze_web_page_views
    .withColumn("page_viewed_at", F.to_timestamp("page_viewed_at"))
    .withColumn("updated_at",      F.to_timestamp("updated_at"))
)

silver_web_page_views = latest_record(web_page_views, "page_view_id").select(
    "page_view_id",
    "session_id",
    "tenant_id",
    "page_path",
    "page_title",
    "referrer",
    "duration_seconds",
    "page_viewed_at",
    "updated_at",
)

(
    silver_web_page_views.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.fact_web_page_view")
)


# ---------------------------------------------------------
# 3. Web Events  (bronze → silver fact table)
# ---------------------------------------------------------

bronze_web_events = spark.table("voyage.bronze.website_events")

web_events = (
    bronze_web_events
    .withColumn("event_timestamp", F.to_timestamp("event_timestamp"))
    .withColumn("event_value",      F.col("event_value").cast("decimal(14,2)"))
    .withColumn("updated_at",        F.to_timestamp("updated_at"))
)

silver_web_events = latest_record(web_events, "event_id").select(
    "event_id",
    "session_id",
    "tenant_id",
    "event_name",
    "event_value",
    "page_path",
    "event_timestamp",
    "updated_at",
)

(
    silver_web_events.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("voyage.silver.fact_web_event")
)


# ---------------------------------------------------------
# Silver table names for verification cells
# (No .cache() on serverless — Delta handles file-level caching)
# ---------------------------------------------------------
silver_table_names = [
    "fact_web_session",
    "fact_web_page_view",
    "fact_web_event",
]


def silver_tbl(name):
    """Helper to read a silver table by short name."""
    return spark.table(f"voyage.silver.{name}")


print("Website silver fact tables created successfully.")

## 2. Silver Layer Data Quality Verification

Ran four quality checks across all three silver fact tables:

* Row counts — confirmed each table loaded with expected volume
* Schema inspection — displayed column names, types, and nullability
* Null checks — validated that key columns (IDs, timestamps, tenant) contained no nulls
* Duplicate key checks — ensured business keys (`session_id`, `page_view_id`, `event_id`) were unique

In [0]:
# ---------------------------------------------------------
# Silver website verification — basic quality checks (optimized)
#   1. Row counts
#   2. Schemas
#   3. Null checks on key columns (combined via agg)
#   4. Duplicate key checks
# Uses silver_tbl() helper from cell 3
# ---------------------------------------------------------

print("=" * 70)
print("1. ROW COUNTS")
print("=" * 70)

for name in silver_table_names:
    cnt = silver_tbl(name).count()
    print(f"  voyage.silver.{name:40s}  {cnt:>10,} rows")

print()
print("=" * 70)
print("2. SCHEMAS")
print("=" * 70)

for name in silver_table_names:
    df = silver_tbl(name)
    print(f"\n  --- voyage.silver.{name} ---")
    for f in df.schema.fields:
        print(f"    {f.name:25s}  {str(f.dataType):20s}  nullable={f.nullable}")

print()
print("=" * 70)
print("3. NULL CHECKS ON KEY COLUMNS")
print("=" * 70)

# Combined null checks: one agg per table instead of one count per column
null_checks = {
    "fact_web_session":   ["session_id", "tenant_id", "session_start"],
    "fact_web_page_view": ["page_view_id", "session_id", "tenant_id", "page_viewed_at"],
    "fact_web_event":     ["event_id", "session_id", "tenant_id", "event_timestamp"],
}

for name, cols in null_checks.items():
    df = silver_tbl(name)
    null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]
    null_counts = df.agg(*null_exprs).collect()[0]
    for col in cols:
        null_cnt = null_counts[col]
        status = "PASS" if null_cnt == 0 else f"FAIL ({null_cnt:,} nulls)"
        print(f"  voyage.silver.{name:40s}  {col:20s}  {status}")

print()
print("=" * 70)
print("4. DUPLICATE KEY CHECKS")
print("=" * 70)

dup_checks = {
    "fact_web_session":   "session_id",
    "fact_web_page_view": "page_view_id",
    "fact_web_event":     "event_id",
}

for name, key in dup_checks.items():
    dup_cnt = (
        silver_tbl(name)
        .groupBy(key)
        .agg(F.count("*").alias("cnt"))
        .filter(F.col("cnt") > 1)
        .count()
    )
    status = "PASS" if dup_cnt == 0 else f"FAIL ({dup_cnt:,} dup keys)"
    print(f"  voyage.silver.{name:40s}  {key:20s}  {status}")

print()
print("Verification complete.")

## 3. Referential Integrity Validation

Checked that foreign-key relationships held across the silver layer with zero expected orphans:

1. `fact_web_session` → `dim_tenant` (on `tenant_id`)
2. `fact_web_page_view` → `fact_web_session` (on `session_id`)
3. `fact_web_event` → `fact_web_session` (on `session_id`)
4. `fact_web_session` → `dim_campaign` (on `campaign_id`, non-null only — organic/direct traffic is excluded)

In [0]:
# ---------------------------------------------------------
# Silver website validation -- referential integrity only
#   Silver = clean, deduplicated analytical source (no KPIs)
# Uses silver_tbl() helper from cell 3
#
# Checks (expected orphan count = 0 for each):
#   1. fact_web_session   -> dim_tenant        (tenant_id)
#   2. fact_web_page_view -> fact_web_session  (session_id)
#   3. fact_web_event     -> fact_web_session  (session_id)
#   4. fact_web_session  -> dim_campaign      (campaign_id, non-null only)
# ---------------------------------------------------------

print("=" * 70)
print("SILVER WEBSITE -- REFERENTIAL INTEGRITY VALIDATION")
print("=" * 70)

all_pass = True

# --- 1. session -> tenant ------------------------------------------
session_tenant_orphans = (
    silver_tbl("fact_web_session")
    .select("tenant_id")
    .distinct()
    .join(
        spark.table("voyage.silver.dim_tenant").select("tenant_id").distinct(),
        "tenant_id",
        "left_anti",
    )
    .count()
)
status_1 = "PASS" if session_tenant_orphans == 0 else f"FAIL ({session_tenant_orphans:,} orphans)"
if session_tenant_orphans != 0:
    all_pass = False
print(f"  1. fact_web_session   -> dim_tenant         {status_1}")

# --- 2. page_view -> session ---------------------------------------
page_view_session_orphans = (
    silver_tbl("fact_web_page_view")
    .select("session_id")
    .distinct()
    .join(
        silver_tbl("fact_web_session").select("session_id").distinct(),
        "session_id",
        "left_anti",
    )
    .count()
)
status_2 = "PASS" if page_view_session_orphans == 0 else f"FAIL ({page_view_session_orphans:,} orphans)"
if page_view_session_orphans != 0:
    all_pass = False
print(f"  2. fact_web_page_view -> fact_web_session    {status_2}")

# --- 3. event -> session -------------------------------------------
event_session_orphans = (
    silver_tbl("fact_web_event")
    .select("session_id")
    .distinct()
    .join(
        silver_tbl("fact_web_session").select("session_id").distinct(),
        "session_id",
        "left_anti",
    )
    .count()
)
status_3 = "PASS" if event_session_orphans == 0 else f"FAIL ({event_session_orphans:,} orphans)"
if event_session_orphans != 0:
    all_pass = False
print(f"  3. fact_web_event     -> fact_web_session    {status_3}")

# --- 4. session campaign_id -> dim_campaign (non-null only) --------
sessions_with_campaign = (
    silver_tbl("fact_web_session")
    .filter(F.col("campaign_id").isNotNull())
)

campaign_orphans = (
    sessions_with_campaign
    .select("campaign_id")
    .distinct()
    .join(
        spark.table("voyage.silver.dim_campaign").select("campaign_id").distinct(),
        "campaign_id",
        "left_anti",
    )
    .count()
)
status_4 = "PASS" if campaign_orphans == 0 else f"FAIL ({campaign_orphans:,} orphans)"
if campaign_orphans != 0:
    all_pass = False
print(f"  4. fact_web_session   -> dim_campaign       {status_4}  (non-null campaign_ids only)")

print("=" * 70)
if all_pass:
    print("RESULT: ALL CHECKS PASSED -- 0 orphans across all 4 relationships")
else:
    print("RESULT: SOME CHECKS FAILED -- review orphan counts above")
print("=" * 70)

## 4. Silver Table Column Listing for dbt Staging

Printed the column names of each silver fact table to help write dbt staging model `SELECT` statements. This kept the staging models in sync with the actual silver schema.

In [0]:
# ---------------------------------------------------------
# Column names for dbt staging references
# Uses silver_tbl() helper from cell 3
# ---------------------------------------------------------

for name in silver_table_names:
    cols = silver_tbl(name).columns
    print(f"-- voyage.silver.{name} ({len(cols)} columns)")
    print(",\n".join(f"    {c}" for c in cols))
    print()